In [1]:
import cv2
import numpy as np
import glob
CHECKERBOARD = (7, 11)

In [2]:
criteria = (cv2.TermCriteria_EPS + cv2.TermCriteria_MAX_ITER, 30, 0.001)

objp = np.zeros((CHECKERBOARD[0]*CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)


In [3]:
objpoints = []
imgpoints = []

In [4]:
images = glob.glob('/content/images*.jpeg')

In [5]:
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    if ret:
        objpoints.append(objp)

        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

        cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)

## Camera Calibration

In [6]:
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None,
    flags=cv2.CALIB_FIX_K3
)

print("\nReprojection Error (ret):", ret)
print("\nCamera Matrix (K):\n", K)
print("\nDistortion Coefficients:\n", dist)


Reprojection Error (ret): 1.8926354764281905

Camera Matrix (K):
 [[1.45959091e+03 0.00000000e+00 5.97884758e+02]
 [0.00000000e+00 1.46229371e+03 4.32471773e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Distortion Coefficients:
 [[ 4.16064277e-01 -2.36264243e+00 -2.60235822e-04 -2.21482875e-03
   0.00000000e+00]]


## Undistroted

In [7]:
img = cv2.imread(images[0])
h, w = img.shape[:2]

newcameramtx, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w,h), 1, (w,h))
dst = cv2.undistort(img, K, dist, None, newcameramtx)

cv2.imwrite('/content/undistorted.jpg', dst)

True

In [8]:
mean_error = 0

for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    mean_error += error

print("\nMean Reprojection Error:", mean_error / len(objpoints))


Mean Reprojection Error: 0.20223144840705432


In [9]:
print(ret)

1.8926354764281905


## Projection Matrix P = K [R | t]


In [10]:
R, _ = cv2.Rodrigues(rvecs[0])
Rt = np.hstack((R, tvecs[0]))

P = np.dot(K, Rt)

print("\nProjection Matrix (P):\n", P)


Projection Matrix (P):
 [[-2.18909716e+02 -1.10398319e+03  1.10506627e+03  2.82915045e+04]
 [ 1.31800429e+03  2.35155110e+02  7.30001032e+02  1.03282388e+04]
 [-2.34382698e-01  3.94346538e-01  8.88569389e-01  2.93223991e+01]]
